# PharmaShield — 1-layer GraphSAGE Shock Propagation Model

This notebook trains a Graph Neural Network (GNN) to predict the risk propagation across the pharmaceutical supply chain in response to regional production shocks (e.g., factory shutdowns in China).

In [ ]:
# Install dependencies for Colab
!pip install torch==2.4.1 torch-geometric==2.6.1 --quiet
!pip install networkx==3.3 --quiet

In [ ]:
import torch
import torch.nn as nn
from torch.optim import Adam
from torch_geometric.nn import SAGEConv
from torch_geometric.data import Data
import networkx as nx
import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime
from sklearn.metrics import roc_auc_score

# Set seed for reproducibility
torch.manual_seed(42)

# Path configuration
DATA_DIR = Path("./data/seed") # Adjust for Colab if needed
WEIGHTS_DIR = Path("./ml/weights")
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def load_json(name):
    with open(DATA_DIR / name, "r") as f:
        return json.load(f)

drugs = load_json("drugs.json")
apis = load_json("apis.json")
dependencies = load_json("dependencies.json")
historical = load_json("historical_disruptions.json")

In [ ]:
# Build Node Map
all_nodes = []
for d in drugs: all_nodes.append((d['id'], 'drug', d))
for a in apis: all_nodes.append((a['id'], 'api', a))

provinces = set()
for a in apis:
    for p in a['primary_provinces']: provinces.add(p)
for p in provinces: all_nodes.append((p, 'province', {}))

id_to_idx = {node[0]: i for i, node in enumerate(all_nodes)}
idx_to_id = {i: node[0] for i, node in enumerate(all_nodes)}

def get_node_features(node_id, node_type, data, is_shocked=0, shock_severity=0, shock_duration=0):
    # Feature vector: [is_drug, is_api, is_province, log_vol, tier1, tier2, tier3, has_sub, china_share, is_shocked, severity, duration]
    feat = [0.0] * 12
    if node_type == 'drug':
        feat[0] = 1.0
        feat[3] = np.log10(max(1000, data.get('patient_population_estimate', 1000))) / 9.0
        tier = data.get('nlem_tier', 'TIER_3')
        if tier == 'TIER_1': feat[4] = 1.0
        elif tier == 'TIER_2': feat[5] = 1.0
        else: feat[6] = 1.0
        feat[7] = 1.0 if data.get('has_substitute') else 0.0
    elif node_type == 'api':
        feat[1] = 1.0
        feat[3] = np.log10(max(1, data.get('monthly_import_value_usd_millions', 0) * 1e6)) / 9.0
        feat[8] = data.get('china_share', 0)
    elif node_type == 'province':
        feat[2] = 1.0
        feat[9] = float(is_shocked)
        feat[10] = float(shock_severity)
        feat[11] = float(shock_duration) / 30.0
    return feat

# Base features
X_base = torch.tensor([get_node_features(n[0], n[1], n[2]) for n in all_nodes], dtype=torch.float)

# Edges
edge_index = torch.tensor([
    [id_to_idx[e['source']], id_to_idx[e['target']]] for e in dependencies
], dtype=torch.long).t().contiguous()

In [ ]:
# Pseudo-label Rule Propagator
nx_graph = nx.DiGraph()
for e in dependencies: nx_graph.add_edge(e['source'], e['target'], weight=e['weight'])

def get_labels(shocked_province, severity):
    labels = torch.zeros(len(all_nodes))
    for i, (node_id, node_type, _) in enumerate(all_nodes):
        if node_type == 'drug':
            risk = 0
            # Find paths drug -> api -> province
            for api in nx_graph.successors(node_id):
                if nx_graph.has_edge(api, shocked_province):
                    w_da = nx_graph[node_id][api]['weight']
                    w_ap = nx_graph[api][shocked_province]['weight']
                    risk += w_da * w_ap * severity
            labels[i] = min(1.0, risk)
    return labels

In [ ]:
# Generate Scenarios
scenarios = []
for _ in range(50):
    prov = np.random.choice(list(provinces))
    sev = np.random.uniform(0.3, 1.0)
    dur = np.random.choice([7, 14, 30])
    
    X = X_base.clone()
    p_idx = id_to_idx[prov]
    X[p_idx, 9] = 1.0
    X[p_idx, 10] = sev
    X[p_idx, 11] = dur / 30.0
    
    y = get_labels(prov, sev)
    scenarios.append(Data(x=X, edge_index=edge_index, y=y))

print(f"Generated {len(scenarios)} training scenarios.")

In [ ]:
class ShockGNN(nn.Module):
    def __init__(self, in_dim=12, hidden=32):
        super().__init__()
        self.conv = SAGEConv(in_dim, hidden, aggr="mean")
        self.head = nn.Linear(hidden, 1)
        
    def forward(self, x, edge_index):
        h = torch.relu(self.conv(x, edge_index))
        return torch.sigmoid(self.head(h)).squeeze(-1)

model = ShockGNN()
optimizer = Adam(model.parameters(), lr=1e-3)
criterion = nn.BCELoss()

# Training Loop
drug_mask = torch.tensor([n[1] == 'drug' for n in all_nodes])
losses = []
model.train()
for epoch in range(100):
    epoch_loss = 0
    for data in scenarios:
        optimizer.zero_grad()
        out = model(data.x, data.edge_index)
        # Only train on drug nodes
        loss = criterion(out[drug_mask], data.y[drug_mask])
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    losses.append(epoch_loss / len(scenarios))
    if epoch % 20 == 0:
        print(f"Epoch {epoch}, Loss: {losses[-1]:.4f}")

plt.plot(losses)
plt.title("GNN Training Loss")
plt.show()

In [ ]:
# Save Model
torch.save(model.state_dict(), WEIGHTS_DIR / "gnn_v1.pt")

metadata = {
    "model_type": "GraphSAGE",
    "version": "1.0",
    "node_feature_dim": 12,
    "hidden_channels": 32,
    "num_layers": 1,
    "id_to_idx": id_to_idx,
    "trained_at": datetime.utcnow().isoformat()
}

with open(WEIGHTS_DIR / "gnn_v1.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("Model and metadata saved.")

### Inference Smoke Test

Predicting risk for a full shutdown in Hebei.

In [ ]:
model.eval()
with torch.no_grad():
    test_X = X_base.clone()
    h_idx = id_to_idx['hebei']
    test_X[h_idx, 9] = 1.0 # is_shocked
    test_X[h_idx, 10] = 1.0 # max severity
    
    risk_probs = model(test_X, edge_index)
    
    drug_risks = []
    for i, (node_id, node_type, _) in enumerate(all_nodes):
        if node_type == 'drug':
            drug_risks.append((node_id, risk_probs[i].item()))
            
    top_5 = sorted(drug_risks, key=lambda x: x[1], reverse=True)[:5]
    print("Top 5 Risky Drugs for Hebei Shutdown:")
    for d, r in top_5:
        print(f"  {d}: {r*100:.1f}%")